In [1]:
# ── 0. Setup ──────────────────────────────────────────────────────────────────
set.seed(114514)

library(tidyverse)
library(haven)

# ── 1. Load & Filter Census Data (chunked for memory efficiency) ──────────────

FILE_PATH   <- "cen_ind_2021_pumf_v2.dta"
CHUNK_SIZE  <- 50000

# Filter criteria (see Census PUMF User Guide for code definitions)
AGE_GROUPS_KEPT  <- c(16, 17)   # 60–64 years, 65–69 years
IMMSTAT_EXCLUDED <- c(3, 88)    # Exclude: non-permanent residents, not applicable
AGEIMM_EXCLUDED  <- 88          # Exclude: not applicable (i.e., born in Canada)
LFACT_EXCLUDED   <- c(14, 88, 99) # Exclude: outside Canada, not applicable, unknown

filtered_chunks <- list()
current_row     <- 0

repeat {
  chunk <- read_dta(FILE_PATH, skip = current_row, n_max = CHUNK_SIZE)

  if (nrow(chunk) == 0) break   # Reached end of file

  rows_to_keep <- which(
    chunk$agegrp %in% AGE_GROUPS_KEPT        &
    !(chunk$immstat %in% IMMSTAT_EXCLUDED)   &
    chunk$ageimm != AGEIMM_EXCLUDED          &
    !(chunk$lfact %in% LFACT_EXCLUDED)
  )

  if (length(rows_to_keep) > 0) {
    filtered_chunks[[length(filtered_chunks) + 1]] <- chunk[rows_to_keep, ]
  }

  current_row <- current_row + CHUNK_SIZE
  cat("Processed up to row:", current_row, "\n")
}

raw_df <- bind_rows(filtered_chunks)
rm(filtered_chunks)
gc()

cat("Loaded", nrow(raw_df), "rows.\n")

# ── 2. Validate Key Variables ─────────────────────────────────────────────────

raw_df |> distinct(agegrp)  |> print(n = Inf)   # Expect: 16, 17
raw_df |> distinct(immstat) |> print(n = Inf)   # Expect: 1 (non-immigrant), 2 (immigrant)
raw_df |> distinct(lfact)   |> print(n = Inf)   # Should exclude 14, 88, 99
raw_df |> distinct(ageimm)  |> arrange(ageimm) |> print(n = Inf)

# ── 3. Create Analysis Variables ──────────────────────────────────────────────

# OAS eligibility requires ~10 years of Canadian residency before age 65.
# Immigrants who arrived at ageimm codes 1–10 (i.e., before age 50) have
# sufficient residency time. Those in the 65–69 group who arrived at code 11
# (ages 50–54) also qualify — they've had 10+ years by their late 60s.

df <- raw_df |>
  mutate(
    # 1 = aged 65–69, 0 = aged 60–64
    over65 = as.integer(agegrp == 17),

    # 1 = not in the labour force (lfact 11–13), 0 = employed or unemployed
    nilf = as.integer(lfact >= 11),

    # 1 = likely OAS-eligible based on residency history
    oas_eligible = case_when(
      immstat == 1                              ~ 1L,  # Non-immigrants: always eligible
      immstat == 2 & ageimm %in% 1:10          ~ 1L,  # Immigrants arrived before age 50
      immstat == 2 & ageimm == 11 & over65 == 1 ~ 1L, # Arrived age 50–54, now 65–69
      TRUE                                      ~ 0L
    )
  )

cat("Final analysis dataset:", nrow(df), "rows.\n")

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Processed up to row: 50000 
Processed up to row: 1e+05 
Processed up to row: 150000 
Processed up to row: 2e+05 
Processed up to row: 250000 
Processed up to row: 3e+05 
Processed up to row: 350000 
Processed up to row: 4e+05 
Processed up to row: 450000 
Processed up to row: 5e+05 
Processed up to row: 550000 
Processed up to row: 6e+05 
Processed up to row: 650000 
Processed up to row: 7e+05 
Processed up to row: 750000 
Processed up to row: 8e+05 
Processed up to row: 850000 
Processed up to row: 9e+05 
Processed up to row: 950000 
Processed up to row: 1e+06 


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1233499,65.9,2438621,130.3,2438621,130.3
Vcells,19211044,146.6,55777344,425.6,55701030,425.0


Loaded 118362 rows.
# A tibble: 2 × 1
  agegrp             
  <dbl+lbl>          
1 16 [60 to 64 years]
2 17 [65 to 69 years]
# A tibble: 2 × 1
  immstat           
  <dbl+lbl>         
1 1 [Non-immigrants]
2 2 [Immigrants]    
# A tibble: 13 × 1
   lfact                                                         
   <dbl+lbl>                                                     
 1 13 [Not in the labour force - Last worked before 2020]        
 2  1                                                            
 3 11 [Not in the labour force - Last worked in 2021]            
 4 12 [Not in the labour force - Last worked in 2020]            
 5  5 [Unemployed - Temporary layoff - Looked for part-time work]
 6  4 [Unemployed - Temporary layoff - Looked for full-time work]
 7  3 [Unemployed - Temporary layoff - Did not look for work]    
 8  2 [Employed - Absent in reference week]                      
 9  6 [Unemployed - New job - Did not look for work]             
10  9 [Unemployed - Looked 

In [2]:
names(df)

[1] "ppsort"           "aboid"            "agegrp"          
  [4] "ageimm"           "attsch"           "bfnmemb"         
  [7] "bedrm"            "CFInc"            "CFInc_AT"        
 [10] "cfstat"           "chdbn"            "CIP2021"         
 [13] "CIP2021_STEM_SUM" "cma"              "condo"           
 [16] "COVID_ERB"        "cow"              "cqppb"           
 [19] "CapGn"            "CfSize"           "ChldC"           
 [22] "CitOth"           "Citizen"          "dist"            
 [25] "dpgrsum"          "dtype"            "EFDecile"        
 [28] "EFInc"            "EFInc_AT"         "eicbn"           
 [31] "ethder"           "EfDIMBM_2018"     "EfSize"          
 [34] "EmpIn"            "fol"              "fptwk"           
 [37] "Gender"           "genstat"          "GovtI"           
 [40] "GTRfs"            "HCORENEED_IND"    "hdgree"          
 [43] "HHInc"            "HHInc_AT"         "hhmrkinc"        
 [46] "hhsize"           "hhtype"           "hlmosten"        
 [49] "hlmostfr"         "hlmostno"         "hlregen"         
 [52] "hlregfr"          "hlregno"          "IMMCAT5"         
 [55] "immstat"          "IncTax"           "Invst"           
 [58] "jobperm"          "kol"              "lfact"           
 [61] "LICO_BT"          "LICO_AT"          "liprogtype"      
 [64] "LI_ELIG_OML_U18"  "locstud"          "LOC_ST_RES"      
 [67] "lstwrk"           "lwmosten"         "lwmostfr"        
 [70] "lwmostno"         "lwregen"          "lwregfr"         
 [73] "lwregno"          "LoLIMA"           "LoLIMB"          
 [76] "LOMBM_2018"       "mode"             "MTNEn"           
 [79] "MTNFr"            "mtnno"            "marsth"          
 [82] "Mob1"             "Mob5"             "MrkInc"          
 [85] "naics"            "NOC21"            "nol"             
 [88] "nos"              "oasgi"            "OtInc"           
 [91] "PKID25"           "PKID0_1"          "PKID15_24"       
 [94] "PKID2_5"          "PKID6_14"         "pkids"           
 [97] "pob"              "POBPAR1"          "POBPAR2"         
[100] "powst"            "pr"               "PR1"             
[103] "PR5"              "presmortg"        "prihm"           
[106] "pwdur"            "pwleave"          "pwocc"           
[109] "pwpr"             "regind"           "relig"           
[112] "repair"           "room"             "Retir"           
[115] "shelco"           "ssgrad"           "subsidy"         
[118] "SempI"            "tenur"            "TotInc"          
[121] "TotInc_AT"        "vismin"           "Value"           
[124] "wkswrk"           "wrkact"           "Wages"           
[127] "yrim"             "weight"           "WT1"             
[130] "WT2"              "WT3"              "WT4"             
[133] "WT5"              "WT6"              "WT7"             
[136] "WT8"              "WT9"              "WT10"            
[139] "WT11"             "WT12"             "WT13"            
[142] "WT14"             "WT15"             "WT16"            
[145] "over65"           "nilf"             "oas_eligible"

In [3]:
#Covariates
df |> select(Gender, hdgree, fol, marsth, pr) |> glimpse()

Rows: 118,362
Columns: 5
$ Gender <dbl+lbl> 2, 1, 1, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, 1, 1, 1, 2, 1, 2, 2,…
$ hdgree <dbl+lbl>  5,  2, 12,  6,  8,  9,  1,  2,  1,  2,  7,  4,  2,  6, 10,…
$ fol    <dbl+lbl> 1, 1, 1, 2, 1, 1, 1, 1, 2, 1, 2, 1, 1, 2, 1, 1, 1, 2, 1, 1,…
$ marsth <dbl+lbl> 2, 6, 2, 3, 2, 2, 5, 2, 3, 3, 2, 1, 2, 3, 2, 6, 4, 2, 2, 2,…
$ pr     <dbl+lbl> 48, 59, 35, 24, 48, 35, 35, 35, 24, 35, 24, 59, 35, 24, 24,…


In [5]:
df |>
  mutate(D = oas_eligible * over65) |>
  select(nilf, oas_eligible, over65, D, Gender, fol, hdgree, marsth, pr) |>
  filter(hdgree != 88, marsth != 8) |>
  summarise(across(everything(), list(
    mean = ~mean(.x, na.rm = TRUE),
    sd   = ~sd(.x, na.rm = TRUE),
    min  = ~min(.x, na.rm = TRUE),
    max  = ~max(.x, na.rm = TRUE)
  ))) |>
  pivot_longer(everything(), 
               names_to = c("variable", "stat"), 
               names_sep = "_(?=[^_]+$)") |>
  pivot_wider(names_from = stat, values_from = value) |> 
  mutate(across(where(is.numeric), ~round(.x, 3)))

Warning message:
“`nilf_mean` and `fol_min` have conflicting value labels.
ℹ Labels for these values will be taken from `nilf_mean`.
✖ Values: 1 and 2”
Warning message:
“`nilf_mean` and `fol_max` have conflicting value labels.
ℹ Labels for these values will be taken from `nilf_mean`.
✖ Values: 1 and 2”
Warning message:
“`nilf_mean` and `hdgree_min` have conflicting value labels.
ℹ Labels for these values will be taken from `nilf_mean`.
✖ Values: 1, 2, 3, and 4”
Warning message:
“`nilf_mean` and `hdgree_max` have conflicting value labels.
ℹ Labels for these values will be taken from `nilf_mean`.
✖ Values: 1, 2, 3, and 4”
Warning message:
“`nilf_mean` and `marsth_min` have conflicting value labels.
ℹ Labels for these values will be taken from `nilf_mean`.
✖ Values: 1, 2, 3, 4, 5, 6, and 8”
Warning message:
“`nilf_mean` and `marsth_max` have conflicting value labels.
ℹ Labels for these values will be taken from `nilf_mean`.
✖ Values: 1, 2, 3, 4, 5, 6, and 8”
Warning message:
“`nilf_mean` 

variable,mean,sd,min,max
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
nilf,0.534,0.499,0,1
oas_eligible,0.980,0.139,0,1
over65,0.453,0.498,0,1
D,0.445,0.497,0,1
Gender,1.491,0.500,1,2
fol,1.317,0.595,1,4
hdgree,4.753,3.403,1,13
marsth,2.630,1.348,1,6
pr,36.095,13.389,10,70


In [6]:
df |>
  group_by(oas_eligible, over65) |>
  summarise(nilf_rate = mean(nilf), n = n())

`summarise()` has grouped output by 'oas_eligible'. You can override using the
`.groups` argument.


oas_eligible,over65,nilf_rate,n
<int>,<int>,<dbl>,<int>
0,0,0.3714075,1357
0,1,0.6419387,1011
1,0,0.4085689,63392
1,1,0.6860956,52602


In [7]:
#DiD = (eligible 65plus - eligible 65 below) - (illegible 65plus - illegible 65 below)
DiD <- (0.6860956 - 0.4085689) - (0.6419387 - 0.3714075)
DiD

[1] 0.0069955

In [9]:
spec1 <- lm(nilf ~ oas_eligible + over65 + oas_eligible:over65, df)
summary(spec1)


Call:
lm(formula = nilf ~ oas_eligible + over65 + oas_eligible:over65, 
    data = df)

Residuals:
    Min      1Q  Median      3Q     Max 
-0.6861 -0.4086  0.3139  0.3139  0.6286 

Coefficients:
                    Estimate Std. Error t value Pr(>|t|)    
(Intercept)         0.371408   0.013013  28.542  < 2e-16 ***
oas_eligible        0.037161   0.013151   2.826  0.00472 ** 
over65              0.270531   0.019915  13.584  < 2e-16 ***
oas_eligible:over65 0.006996   0.020115   0.348  0.72800    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.4794 on 118358 degrees of freedom
Multiple R-squared:  0.07678,	Adjusted R-squared:  0.07676 
F-statistic:  3281 on 3 and 118358 DF,  p-value: < 2.2e-16


In [10]:
spec2 <- lm(nilf ~ oas_eligible + over65 + oas_eligible:over65 + Gender + marsth + hdgree, df)
summary(spec2)


Call:
lm(formula = nilf ~ oas_eligible + over65 + oas_eligible:over65 + 
    Gender + marsth + hdgree, data = df)

Residuals:
    Min      1Q  Median      3Q     Max 
-0.7738 -0.4446  0.2375  0.3908  1.1504 

Coefficients:
                      Estimate Std. Error t value Pr(>|t|)    
(Intercept)          0.5916382  0.0140010  42.257  < 2e-16 ***
oas_eligible         0.0282431  0.0130386   2.166   0.0303 *  
over65               0.2675712  0.0197389  13.556  < 2e-16 ***
Gender              -0.1131480  0.0027839 -40.644  < 2e-16 ***
marsth              -0.0055395  0.0010234  -5.413 6.22e-08 ***
hdgree              -0.0057352  0.0002563 -22.379  < 2e-16 ***
oas_eligible:over65  0.0107463  0.0199358   0.539   0.5899    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.4751 on 118355 degrees of freedom
Multiple R-squared:  0.09319,	Adjusted R-squared:  0.09314 
F-statistic:  2027 on 6 and 118355 DF,  p-value: < 2.2e-16


In [11]:
spec3 <- lm(nilf ~ oas_eligible + over65 + oas_eligible:over65 + Gender + marsth + hdgree + pr, df)
summary(spec3)


Call:
lm(formula = nilf ~ oas_eligible + over65 + oas_eligible:over65 + 
    Gender + marsth + hdgree + pr, data = df)

Residuals:
    Min      1Q  Median      3Q     Max 
-0.8123 -0.4460  0.2303  0.3944  1.1609 

Coefficients:
                      Estimate Std. Error t value Pr(>|t|)    
(Intercept)          0.6551004  0.0146221  44.802  < 2e-16 ***
oas_eligible         0.0203348  0.0130373   1.560    0.119    
over65               0.2661757  0.0197207  13.497  < 2e-16 ***
Gender              -0.1134509  0.0027814 -40.790  < 2e-16 ***
marsth              -0.0056719  0.0010225  -5.547 2.91e-08 ***
hdgree              -0.0056376  0.0002561 -22.011  < 2e-16 ***
pr                  -0.0015370  0.0001032 -14.899  < 2e-16 ***
oas_eligible:over65  0.0123696  0.0199176   0.621    0.535    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.4746 on 118354 degrees of freedom
Multiple R-squared:  0.09488,	Adjusted R-squared:  0.09483 
F-statistic:  1

In [12]:
spec4 <- lm(nilf ~ oas_eligible + over65 + oas_eligible:over65 + Gender + marsth + hdgree + pr + fol, df)
summary(spec4)


Call:
lm(formula = nilf ~ oas_eligible + over65 + oas_eligible:over65 + 
    Gender + marsth + hdgree + pr + fol, data = df)

Residuals:
    Min      1Q  Median      3Q     Max 
-0.8660 -0.4378  0.2245  0.3980  1.1588 

Coefficients:
                      Estimate Std. Error t value Pr(>|t|)    
(Intercept)          0.5763639  0.0158677  36.323  < 2e-16 ***
oas_eligible         0.0416753  0.0131359   3.173  0.00151 ** 
over65               0.2528366  0.0197352  12.811  < 2e-16 ***
Gender              -0.1133921  0.0027795 -40.796  < 2e-16 ***
marsth              -0.0057810  0.0010219  -5.657 1.54e-08 ***
hdgree              -0.0054144  0.0002566 -21.105  < 2e-16 ***
pr                  -0.0011076  0.0001085 -10.211  < 2e-16 ***
fol                  0.0316945  0.0024901  12.728  < 2e-16 ***
oas_eligible:over65  0.0252579  0.0199298   1.267  0.20503    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.4743 on 118353 degrees of freedom
Multip

In [13]:
tibble(
  Specification = c("Spec 1", "Spec 2", "Spec 3", "Spec 4"),
  Coefficient   = c(
    coef(spec1)["oas_eligible:over65"],
    coef(spec2)["oas_eligible:over65"],
    coef(spec3)["oas_eligible:over65"],
    coef(spec4)["oas_eligible:over65"]
  )
) |> mutate(Coefficient = round(Coefficient, 4))

Specification,Coefficient
<chr>,<dbl>
Spec 1,0.0070
Spec 2,0.0107
Spec 3,0.0124
Spec 4,0.0253
